# 02 - Data Cleaning and Feature Engineering

This notebook implements a strict, reproducible data preparation pipeline for 30-day readmission analytics.

**Business alignment**
- Readmission risk identification
- Operational insight dimensions: bed/discharge/admission/service utilization patterns
- Analysis-ready dataset for downstream modeling and reporting

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 0) Load raw dataset and define helper functions

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

preferred_raw_path = PROJECT_ROOT / "data" / "raw" / "hospital_readmissions.csv"
fallback_raw_path = PROJECT_ROOT / "data" / "raw" / "diabetic_data.csv"

if preferred_raw_path.exists():
    raw_path = preferred_raw_path
    source_used = "preferred_hospital_readmissions"
elif fallback_raw_path.exists():
    raw_path = fallback_raw_path
    source_used = "fallback_diabetic_data"
else:
    raise FileNotFoundError(
        "Neither data/raw/hospital_readmissions.csv nor data/raw/diabetic_data.csv was found."
    )

print(f"Using raw file ({source_used}): {raw_path}")
df = pd.read_csv(raw_path)
print("Initial shape:", df.shape)

def standardize_column_names(columns):
    clean = (
        pd.Index(columns)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )
    return clean

def get_col_name(frame, candidates):
    for c in candidates:
        if c in frame.columns:
            return c
    return None


## 1) Data Cleaning

Pipeline:
1. Standardize names
2. Normalize missing markers and inspect missingness
3. Drop columns with >40% missing
4. Impute numeric/categorical values
5. Remove duplicates
6. Fix data types
7. Log outliers using IQR (without blind dropping)

In [ ]:
# 1.1 Standardize column names
before_cols = df.columns.tolist()
df.columns = standardize_column_names(df.columns)
after_cols = df.columns.tolist()

print("Column names standardized.")
display(pd.DataFrame({"before": before_cols, "after": after_cols}).head(20))

# 1.2 Normalize missing markers
df_before_missing = df.copy()
df = df.replace({"?": np.nan, "None": np.nan, "none": np.nan, "": np.nan})

missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_report = missing_pct.to_frame("missing_pct")
display(missing_report.head(20))

# 1.3 Drop columns >40% missing
drop_cols = missing_pct[missing_pct > 40].index.tolist()
print("Columns dropped for >40% missing:", drop_cols if drop_cols else "None")
df = df.drop(columns=drop_cols)

# 1.4 Data type correction attempts for numeric-like columns
for col in df.columns:
    if df[col].dtype == "object":
        converted = pd.to_numeric(df[col], errors="coerce")
        # Convert if at least 90% of non-null values can be numeric.
        non_null = df[col].notna().sum()
        numeric_non_null = converted.notna().sum()
        if non_null > 0 and (numeric_non_null / non_null) >= 0.9:
            df[col] = converted

# 1.5 Date conversion if date-like columns exist
date_like_cols = [c for c in df.columns if any(k in c for k in ["date", "dt", "timestamp", "time_stamp"]) and df[c].dtype == "object"]
for c in date_like_cols:
    parsed = pd.to_datetime(df[c], errors="coerce")
    if parsed.notna().mean() >= 0.6:
        df[c] = parsed

# 1.6 Imputation
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in df.columns if c not in numeric_cols and not pd.api.types.is_datetime64_any_dtype(df[c])]

for col in numeric_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    if df[col].isna().any():
        mode_series = df[col].mode(dropna=True)
        fill_value = mode_series.iloc[0] if len(mode_series) > 0 else "unknown"
        if pd.isna(fill_value):
            fill_value = "unknown"
        df[col] = df[col].fillna(fill_value)

# 1.7 Remove duplicates
before_dupes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after_dupes = len(df)
print(f"Duplicates removed: {before_dupes - after_dupes}")

# 1.8 Outlier logging using IQR (no automatic dropping)
iqr_logs = []
for col in df.select_dtypes(include=[np.number]).columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        continue
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_count = int(((df[col] < lower) | (df[col] > upper)).sum())
    iqr_logs.append({
        "column": col,
        "q1": float(q1),
        "q3": float(q3),
        "iqr": float(iqr),
        "lower_bound": float(lower),
        "upper_bound": float(upper),
        "outlier_count": outlier_count
    })

iqr_log_df = pd.DataFrame(iqr_logs).sort_values("outlier_count", ascending=False)
print("Top IQR outlier counts:")
display(iqr_log_df.head(15))

### Before/After cleaning quality checks

In [ ]:
quality_check = pd.DataFrame({
    "metric": ["rows", "columns", "missing_values_total"],
    "before": [df_before_missing.shape[0], df_before_missing.shape[1], int(df_before_missing.isna().sum().sum())],
    "after": [df.shape[0], df.shape[1], int(df.isna().sum().sum())]
})
display(quality_check)
display(df.head())

## 2) Target variable creation

`readmitted_30_days = 1` when readmitted is `"<30"`, else `0`.

In [ ]:
readmit_col = get_col_name(df, ["readmitted"])
if readmit_col is None:
    raise ValueError("Required readmission column not found (expected `readmitted`).")

readmit_values = df[readmit_col].astype(str).str.strip().str.lower()

if (readmit_values == "<30").any():
    df["readmitted_30_days"] = (readmit_values == "<30").astype(int)
    target_rule_used = "<30 => 1, otherwise 0"
elif set(readmit_values.unique()).issubset({"yes", "no"}):
    # Dataset variant where readmission is already binary.
    df["readmitted_30_days"] = (readmit_values == "yes").astype(int)
    target_rule_used = "yes => 1, no => 0 (schema fallback)"
else:
    raise ValueError(
        "Unsupported readmitted values for target creation. Expected '<30' or yes/no schema."
    )

print("Target rule used:", target_rule_used)
display(df[[readmit_col, "readmitted_30_days"]].head(10))
print(df["readmitted_30_days"].value_counts(dropna=False))


## 3) Feature engineering

Features are engineered with schema-safe logic so this notebook can run on closely related hospital readmission schemas.

In [ ]:
# Resolve variant names used by different versions of the dataset.
col_outpatient = get_col_name(df, ["number_outpatient", "n_outpatient"])
col_inpatient = get_col_name(df, ["number_inpatient", "n_inpatient"])
col_emergency = get_col_name(df, ["number_emergency", "n_emergency"])
col_time_hospital = get_col_name(df, ["time_in_hospital"])
col_lab = get_col_name(df, ["num_lab_procedures", "n_lab_procedures"])
col_proc = get_col_name(df, ["num_procedures", "n_procedures"])
col_meds = get_col_name(df, ["num_medications", "n_medications"])
col_diag_count = get_col_name(df, ["number_diagnoses", "n_diagnoses"])
col_adm_type = get_col_name(df, ["admission_type_id"])
col_discharge = get_col_name(df, ["discharge_disposition_id"])
col_age = get_col_name(df, ["age"])

# PATIENT HISTORY
for c in [col_outpatient, col_inpatient, col_emergency]:
    if c is None:
        raise ValueError("Expected outpatient/inpatient/emergency visit columns are missing.")

df["total_visits"] = df[col_outpatient] + df[col_inpatient] + df[col_emergency]
frequent_threshold = df["total_visits"].quantile(0.75)
df["is_frequent_patient"] = (df["total_visits"] > frequent_threshold).astype(int)

# STAY & UTILIZATION
if col_time_hospital is not None and col_lab is not None:
    df["stay_intensity"] = df[col_time_hospital] * df[col_lab]
else:
    df["stay_intensity"] = 0

if col_time_hospital is not None and col_proc is not None:
    df["procedures_per_day"] = np.where(df[col_time_hospital] > 0, df[col_proc] / df[col_time_hospital], 0)
else:
    df["procedures_per_day"] = 0

# MEDICAL COMPLEXITY
if col_diag_count is not None:
    df["comorbidity_score"] = df[col_diag_count]
else:
    diag_cols = [c for c in ["diag_1", "diag_2", "diag_3"] if c in df.columns]
    df["comorbidity_score"] = df[diag_cols].notna().sum(axis=1) if diag_cols else 0

if col_meds is not None:
    df["meds_per_visit"] = np.where(df["total_visits"] > 0, df[col_meds] / df["total_visits"], df[col_meds])
else:
    df["meds_per_visit"] = 0

# ADMISSION FEATURES
if col_adm_type is not None:
    df["is_emergency_admission"] = (df[col_adm_type].astype(str).isin(["1", "2", "7"])) .astype(int)
    df["is_urgent_admission"] = (df[col_adm_type].astype(str).isin(["2", "7"])) .astype(int)
    admission_risk_map = {"1": 3, "2": 2, "3": 1, "4": 1, "5": 2, "6": 2, "7": 2, "8": 1}
    df["admission_risk_score"] = df[col_adm_type].astype(str).map(admission_risk_map).fillna(1).astype(int)
else:
    df["is_emergency_admission"] = 0
    df["is_urgent_admission"] = 0
    df["admission_risk_score"] = 1

# DISCHARGE FEATURES
if col_discharge is not None:
    discharge_str = df[col_discharge].astype(str)
    high_risk = {"11", "13", "14", "19", "20", "21"}
    moderate_risk = {"2", "3", "4", "5", "6", "7", "8", "9", "10", "12", "15", "16", "17", "18", "22", "23", "24", "25", "26", "27", "28"}

    df["discharge_risk"] = np.select(
        [discharge_str.isin(high_risk), discharge_str.isin(moderate_risk)],
        ["high", "moderate"],
        default="low"
    )
    df["is_home_discharge"] = discharge_str.isin(["1", "6", "8"]).astype(int)
else:
    df["discharge_risk"] = "unknown"
    df["is_home_discharge"] = 0

# DEPARTMENT / SERVICE
lab_component = df[col_lab] if col_lab is not None else 0
proc_component = df[col_proc] if col_proc is not None else 0
med_component = df[col_meds] if col_meds is not None else 0
df["service_utilization_score"] = lab_component * 0.4 + proc_component * 0.3 + med_component * 0.3
df["service_utilization_score"] = df["service_utilization_score"].round(2)

# TEMPORAL FEATURE PROXY
if col_time_hospital is not None:
    df["length_of_stay_bucket"] = pd.cut(
        df[col_time_hospital],
        bins=[-np.inf, 3, 7, np.inf],
        labels=["short", "medium", "long"]
    ).astype(str)
else:
    df["length_of_stay_bucket"] = "unknown"

# DEMOGRAPHIC TRANSFORMS
if col_age is not None:
    age_extract = df[col_age].astype(str).str.extract(r"\[(\d+)-?(\d+)?\)")
    low = pd.to_numeric(age_extract[0], errors="coerce")
    high = pd.to_numeric(age_extract[1], errors="coerce")
    midpoint = np.where(high.notna(), (low + high) / 2, low)
    numeric_age = pd.Series(midpoint, index=df.index).fillna(pd.to_numeric(df[col_age], errors="coerce"))
    df["age_numeric"] = numeric_age.fillna(numeric_age.median())

    df["age_group"] = pd.cut(
        df["age_numeric"],
        bins=[-np.inf, 30, 45, 60, 75, np.inf],
        labels=["young", "adult", "middle_age", "senior", "elderly"]
    ).astype(str)
    df["is_senior"] = (df["age_numeric"] > 60).astype(int)
else:
    df["age_numeric"] = 0
    df["age_group"] = "unknown"
    df["is_senior"] = 0

engineered_cols = [
    "total_visits", "is_frequent_patient", "stay_intensity", "procedures_per_day",
    "comorbidity_score", "meds_per_visit", "is_emergency_admission", "is_urgent_admission",
    "admission_risk_score", "discharge_risk", "is_home_discharge", "service_utilization_score",
    "length_of_stay_bucket", "age_group", "is_senior", "age_numeric"
]

display(df[engineered_cols].head(10))

## 4) Encoding preparation

Label encode low-cardinality categorical columns (<=10 unique values), excluding explicit high-cardinality ID-like columns.

In [ ]:
encoding_map = {}
candidate_cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

high_cardinality_exclusions = []
for col in candidate_cat_cols:
    nunique = df[col].nunique(dropna=False)
    if nunique <= 10:
        categories = sorted(df[col].astype(str).unique().tolist())
        mapper = {cat: idx for idx, cat in enumerate(categories)}
        df[col] = df[col].astype(str).map(mapper)
        encoding_map[col] = mapper
    else:
        high_cardinality_exclusions.append(col)

print("Low-cardinality columns encoded:", list(encoding_map.keys()))
print("High-cardinality columns left untouched:", high_cardinality_exclusions)

## 5) Final dataset checks and save

In [ ]:
# Ensure no missing values remain
remaining_missing = int(df.isna().sum().sum())
print("Remaining missing values:", remaining_missing)

if remaining_missing > 0:
    # Final safety net: numeric -> median, object -> unknown
    for col in df.columns:
        if df[col].isna().any():
            if pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].fillna(df[col].median())
            else:
                df[col] = df[col].fillna("unknown")

print("Missing after final safety net:", int(df.isna().sum().sum()))

processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
output_path = processed_dir / "cleaned_hospital_readmissions.csv"
df.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to: {output_path}")

log_dir = PROJECT_ROOT / "reports" / "data_quality"
log_dir.mkdir(parents=True, exist_ok=True)

iqr_log_path = log_dir / "hospital_readmissions_iqr_outlier_log.csv"
encoding_log_path = log_dir / "hospital_readmissions_encoding_map_summary.csv"

iqr_log_df.to_csv(iqr_log_path, index=False)
pd.DataFrame([
    {"column": k, "num_levels": len(v), "levels": str(list(v.keys())[:20])}
    for k, v in encoding_map.items()
]).to_csv(encoding_log_path, index=False)

print(f"Saved outlier log to: {iqr_log_path}")
print(f"Saved encoding summary to: {encoding_log_path}")

## 6) Final review snapshot

In [ ]:
print("Final shape:", df.shape)
display(df.head(10))
display(df.dtypes.to_frame("dtype").head(30))